# 03. Ephemeris O-C Analysis

This notebook demonstrates how to generate Observed-minus-Calculated (O-C) diagrams from transit timing data. We'll fit a linear ephemeris to the transit midpoints and analyze the residuals to look for potential Transit Timing Variations (TTVs).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.optimize import curve_fit

# --- Configuration ---
TARGET_NAME = "WASP-103b"
# Example initial ephemeris (from exoplanet archive or previous fit)
INITIAL_T0 = 2456950.81156 # BJD_TDB
INITIAL_PERIOD = 0.9255462 # Days

print(f"Starting O-C Analysis for {TARGET_NAME}...")


## 1. Generate Mock Transit Timing Data

In a real scenario, this data would come from transit fitting results stored in `muscat.db`. For this demo, we'll create synthetic observed transit midpoints (Tc) with some scatter and a subtle TTV signal.

In [ ]:
# Generate epochs
epochs = np.arange(0, 50, 1) # 50 transits

# Calculate expected transit times based on initial ephemeris
expected_tc = INITIAL_T0 + epochs * INITIAL_PERIOD

# Add some random noise (observational scatter)
observational_noise = np.random.normal(0, 0.0005, len(epochs)) # ~40 seconds scatter

# Add a synthetic TTV signal (e.g., sinusoidal)
ttv_amplitude = 0.005 # days (approx 7 minutes)
ttv_period_epochs = 15 # TTV period in terms of number of transits
synthetic_ttv = ttv_amplitude * np.sin(2 * np.pi * epochs / ttv_period_epochs)

observed_tc = expected_tc + observational_noise + synthetic_ttv
tc_uncertainties = np.random.normal(0.0003, 0.00005, len(epochs)) # Typical uncertainties

data = {
    'epoch': epochs,
    'observed_tc': observed_tc,
    'tc_uncertainty': tc_uncertainties
}
ttv_df = pd.DataFrame(data)

print("Mock transit timing data generated:")
print(ttv_df.head())


## 2. Linear Ephemeris Fit

We'll fit a linear ephemeris (`Tc = T0 + E * P`) to the observed transit times. This will give us the best-fit linear period (P) and reference epoch (T0).

In [ ]:
def linear_ephemeris(epoch, t0, period):
    return t0 + epoch * period

# Perform the fit, weighting by uncertainties
sigma = ttv_df['tc_uncertainty']
popt, pcov = curve_fit(linear_ephemeris, ttv_df['epoch'], ttv_df['observed_tc'],
                       sigma=sigma, absolute_sigma=True, 
                       p0=[INITIAL_T0, INITIAL_PERIOD]) # Use initial values as guess

fit_t0, fit_period = popt
fit_t0_err, fit_period_err = np.sqrt(np.diag(pcov))

print(f"Fitted T0: {fit_t0:.6f} +/- {fit_t0_err:.6f} BJD_TDB")
print(f"Fitted Period: {fit_period:.7f} +/- {fit_period_err:.7f} Days")

ttv_df['calculated_tc'] = linear_ephemeris(ttv_df['epoch'], fit_t0, fit_period)
ttv_df['oc_residuals'] = (ttv_df['observed_tc'] - ttv_df['calculated_tc']) * 24 * 60 # Convert to minutes


## 3. Plot the O-C Diagram

The O-C diagram plots the residuals (observed minus calculated transit times) against epoch or BJD. Deviations from zero can indicate a TTV signal.

In [ ]:
plt.figure(figsize=(12, 6))
plt.errorbar(ttv_df['epoch'], ttv_df['oc_residuals'], yerr=ttv_df['tc_uncertainty'] * 24 * 60,
             fmt='o', capsize=3, label='O-C Residuals')

plt.axhline(0, color='red', linestyle='--', alpha=0.7, label='Zero O-C')
plt.xlabel("Epoch Number (E)")
plt.ylabel("O-C (Observed - Calculated) [minutes]")
plt.title(f"O-C Diagram for {TARGET_NAME}")
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

print("\nInterpretation: A flat O-C diagram indicates a constant period. Any significant deviation from zero (especially periodic or parabolic patterns) suggests Transit Timing Variations, possibly due to a perturbing body or tidal interactions.")


## 4. Summary

This notebook demonstrated the basic steps to perform an O-C analysis, including generating synthetic data, fitting a linear ephemeris, and plotting the residuals. This forms the basis for more advanced TTV analyses using tools like `harmonic`.